In [88]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import requests

In [89]:
df = pd.read_csv("phrase_classification_10000.csv")
df.head()

,Text,Token_Count,Special_Char_Ratio,Verb_Density,Avg_Word_Length,Is_Identifier_Style,Label
0,Evaluating Database requires a robust Optimiza...,11,0.013,0.0,6.27,0,Semantic
1,The RLHF layer applies Few-shot to the text su...,13,0.029,0.0,4.75,0,Semantic
2,sourcesEncodingEnables,1,0.000,0.0,22.00,1,Syntactic
3,ThatEnablesSoftmax,1,0.000,0.0,18.00,1,Syntactic
4,incorporate_augmented,1,0.048,0.0,21.00,1,Syntactic


In [55]:
import requests
import numpy as np

EMBEDDING_DIM = 384
ollama_url = "http://localhost:11434/api/embeddings"
model = "all-minilm"  # must be pulled first: ollama pull all-minilm

def get_ollama_embedding(text):
    """Calls local Ollama embeddings API."""
    payload = {
        "model": model,
        "prompt": text,
    }
    try:
        response = requests.post(ollama_url, json=payload)
        response.raise_for_status()
        return response.json()["embedding"]
    except Exception as e:
        print(f"Error fetching embedding for: {text[:20]}... -> {e}")
        return [0.0] * EMBEDDING_DIM

# Manual features
X_manual = df[['Token_Count', 'Special_Char_Ratio', 'Verb_Density',
               'Avg_Word_Length', 'Is_Identifier_Style']].values

# Embeddings
print(f"Fetching embeddings from Ollama using '{model}'...")
embeddings_list = [get_ollama_embedding(text) for text in df['Text']]
X_embeddings = np.array(embeddings_list)

# Validate shapes before combining
assert X_embeddings.shape == (len(df), EMBEDDING_DIM), \
    f"Unexpected embedding shape: {X_embeddings.shape}"

X = np.hstack((X_manual, X_embeddings))
print(f"Final feature matrix shape: {X.shape}")

Fetching embeddings from Ollama using 'all-minilm'...
Final feature matrix shape: (10000, 389)


In [90]:
col_names = ['Token_Count','Special_Char_Ratio','Verb_Density','Avg_Word_Length','Is_Identifier_Style'] \
            + [f'emb_{i}' for i in range(X.shape[1] - 5)]
pd.DataFrame(X, columns=col_names).head()

,Token_Count,Special_Char_Ratio,Verb_Density,Avg_Word_Length,Is_Identifier_Style
0,11.0,0.013,0.0,6.27,0.0
1,13.0,0.029,0.0,4.75,0.0
2,1.0,0.000,0.0,22.00,1.0
3,1.0,0.000,0.0,18.00,1.0
4,1.0,0.048,0.0,21.00,1.0


In [98]:
y = df['Label']
X = df.drop(['Text', 'Label'], axis=1).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.99, random_state = 99)

clf = DecisionTreeClassifier(random_state=34)
clf.fit(X_train, y_train)

import pickle

with open("decision_tree_clf.pkl", "wb") as f:
    pickle.dump(clf, f)

print("Classifier saved → decision_tree_clf.pkl")

Classifier saved → decision_tree_clf.pkl


In [93]:
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")  # 4 decimals

Accuracy: 1.0000


In [94]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = DecisionTreeClassifier(max_depth=10, min_samples_leaf=5, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1       : {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1       : 1.0000
              precision    recall  f1-score   support

    Semantic       1.00      1.00      1.00      1540
   Syntactic       1.00      1.00      1.00       460

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

